In [ ]:
import os
import sys
import torch
from pathlib import Path
from tqdm import tqdm

ROOT_DIR = "/root/private_data/luog/codex/IgGM2"
os.chdir(ROOT_DIR)
if ROOT_DIR not in sys.path:
    sys.path.insert(0, ROOT_DIR)

print(f"当前工作目录已切换为: {os.getcwd()}")


from src.iggm_lightning.data_module import ProcessedSabdabDataModule
from IgGM.model.arch.core.diffuser import Diffuser


def calculate_translation_stats():
    # 从 test_debug.yaml 中提取的数据路径
    metadata_path = "/root/private_data/luog/codex/IgGM2/data/sabdab/sabdab/metadata.json"
    pdb_dir = "/root/private_data/luog/codex/IgGM2/data/sabdab/pdb"
    samples_dir = "/root/private_data/luog/codex/IgGM2/data/sabdab/sabdab/samples"
    train_ids_path = "/root/private_data/luog/codex/IgGM2/data/sabdab/sabdab_file/split/train_prot_ids.txt"

    print("初始化 DataModule ...")
    # 初始化 DataModule，num_workers=0 避免多进程卡死
    datamodule = ProcessedSabdabDataModule(
        metadata_path=metadata_path,
        pdb_dir=pdb_dir,
        train_ids_path=train_ids_path,
        samples_dir=samples_dir,
        batch_size=1, 
        num_workers=2, 
    )
    datamodule.setup(stage="fit")
    train_loader = datamodule.train_dataloader()

    all_trsl = []

    print(f"开始遍历训练集提取物理平移量 (trsl_orig_physical)...")
    # 使用 CPU 进行提取以防显存 OOM
    device = torch.device("cpu")
    
    for item in tqdm(train_loader):
        try:
            # data_module 返回字典中 payload 包含了 prot_data_curr
            prot_data = item["payload"]["prot_data_curr"]
            
            # 将需要计算的张量迁移到 CPU
            cord_tns_orig = prot_data["cords_atom14"].to(device)
            cmsk_mat_orig14 = prot_data["cmsk_atom14"].to(device)
            antibody_mask = prot_data["mask_ab"].to(device)

            # 调用 Diffuser 的静态方法计算抗体刚体参数
            _, trsl_orig_physical, _ = Diffuser._build_antibody_rigid_params(
                cord_tns_orig,
                cmsk_mat_orig14,
                antibody_mask
            )
            
            # 收集平移向量 (3D)
            all_trsl.append(trsl_orig_physical.clone().detach())
            
        except Exception as e:
            # 跳过异常或损坏的数据
            print(f"\n警告：跳过样本 {item.get('prot_id')}, 原因: {e}")
            continue

    if not all_trsl:
        print("错误：未提取到任何有效的平移向量。请检查路径或数据是否正确。")
        return

    # 将列表堆叠为 Tensor，形状为 [N, 3]
    all_trsl_tensor = torch.stack(all_trsl).float()

    # 1. 计算均值 (3D 向量)
    trsl_mu = all_trsl_tensor.mean(dim=0)

    # 2. 计算尺度 (标量)
    # 计算所有三维坐标的全局标准差，用于保证归一化后的数据方差为 1.0
    trsl_scale_scalar = all_trsl_tensor.std()

    print("\n" + "="*65)
    print("✅ 计算完成！请将以下参数直接替换到 diffuser2.py 的 __init__ 方法中：")
    print("-" * 65)
    print(f"self.trsl_mu = torch.tensor([{trsl_mu[0]:.4f}, {trsl_mu[1]:.4f}, {trsl_mu[2]:.4f}], dtype=torch.float32)")
    print(f"self.trsl_scale = torch.tensor({trsl_scale_scalar:.4f}, dtype=torch.float32)")
    print("=" * 65)

if __name__ == "__main__":
    calculate_translation_stats()